# Murphy 1 · Chapter 9 · Practice

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app/#M9). Each exercise has a question, a hint, and an answer that stays hidden until you click.

**How to use:** try the starter cell first (fill in the `...` TODOs), then compare with the hidden solution cell (double-click its title to see the code), then open the answer.

## 9.1 · Naive Bayes spam filter from scratch  
*Section §9.3 · code*

Train a naive Bayes classifier on six tiny emails (3 spam, 3 ham) using word counts with add-one (Laplace) smoothing. Classify 'win money now' and 'meeting tomorrow'. Print the log-odds for each.

<details><summary><b>Hint</b></summary>

log P(spam | words) ∝ log P(spam) + Σ_w log P(w | spam). With add-one smoothing, P(w | c) = (count + 1) / (total words in c + vocabulary size).

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
spam = ["win money now", "win a prize now", "cheap money win"]
ham = ["meeting at noon", "project meeting tomorrow", "lunch tomorrow at noon"]
vocab = sorted({w for d in spam + ham for w in d.split()})
def counts(docs): return np.array([sum(d.split().count(w) for d in docs) for w in vocab])
cs, ch = counts(spam), counts(ham)
ps = ...  # TODO: add-one smoothed P(word | spam)
ph = ...  # TODO: add-one smoothed P(word | ham)
def log_odds(text):
    lo = np.log(0.5) - np.log(0.5)
    for w in text.split():
        if w in vocab:
            i = vocab.index(w); lo += ...  # TODO: each word adds its own evidence
    return lo
for t in ["win money now", "meeting tomorrow"]:
    print(f"{t!r:22s} log-odds spam:ham = {log_odds(t):+.3f}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
spam = ["win money now", "win a prize now", "cheap money win"]
ham = ["meeting at noon", "project meeting tomorrow", "lunch tomorrow at noon"]
vocab = sorted({w for d in spam + ham for w in d.split()})
def counts(docs): return np.array([sum(d.split().count(w) for d in docs) for w in vocab])
cs, ch = counts(spam), counts(ham)
ps = (cs + 1) / (cs.sum() + len(vocab))  # add-one smoothed P(word | spam)
ph = (ch + 1) / (ch.sum() + len(vocab))  # add-one smoothed P(word | ham)
def log_odds(text):
    lo = np.log(0.5) - np.log(0.5)
    for w in text.split():
        if w in vocab:
            i = vocab.index(w); lo += np.log(ps[i]) - np.log(ph[i])  # each word adds its own evidence
    return lo
for t in ["win money now", "meeting tomorrow"]:
    print(f"{t!r:22s} log-odds spam:ham = {log_odds(t):+.3f}")

<details><summary><b>Answer and explanation</b></summary>

'win money now' gets a large positive log-odds (spam), 'meeting tomorrow' a negative one (ham). 'Naive' means each word is treated as independent given the class, so the probabilities multiply (logs add). Add-one smoothing stops an unseen word from giving probability 0, which would wipe out everything else (Murphy §9.3.2–9.3.3).

Expected output:
```
'win money now'        log-odds spam:ham = +3.584
'meeting tomorrow'     log-odds spam:ham = -2.197
```

</details>

## 9.2 · Why a shared covariance gives a straight boundary  
*Section §9.2 · derive*

Two classes are Gaussian with means μ₀, μ₁ and the same covariance Σ. Show that log p(x|1) − log p(x|0) is linear in x.

<details><summary><b>Hint</b></summary>

Expand −½(x − μ)ᵀΣ⁻¹(x − μ) = −½xᵀΣ⁻¹x + μᵀΣ⁻¹x − ½μᵀΣ⁻¹μ for each class and subtract.

</details>

✍️ *Write your answer here (double-click to edit).*

<details><summary><b>Worked answer</b></summary>

Both classes contribute the same −½xᵀΣ⁻¹x term (same Σ), and it cancels in the difference. What remains is (μ₁ − μ₀)ᵀΣ⁻¹x + constant: a weighted sum of x plus a bias. So the log-odds is wᵀx + b with w = Σ⁻¹(μ₁ − μ₀). That's exactly logistic regression's form (Murphy §9.2.3). With different covariances the quadratic terms don't cancel and the boundary curves (QDA).

</details>

## 9.3 · Gaussian classifier on real flowers  
*Section §9.2 · code*

On the iris dataset (150 flowers, 4 measurements, 3 species), fit a Gaussian discriminant classifier from scratch: per-class means, a shared covariance (LDA), and class priors. Report test accuracy on a 100/50 split, then compare with the nearest-centroid classifier (same means, identity covariance).

<details><summary><b>Hint</b></summary>

Score each class with −½(x − μ_c)ᵀΣ⁻¹(x − μ_c) + log π_c and pick the largest.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
from sklearn.datasets import load_iris
X, y = load_iris(return_X_y=True)
idx = np.random.default_rng(0).permutation(150); tr, te = idx[:100], idx[100:]
mus = np.array([X[tr][y[tr] == c].mean(0) for c in range(3)])
S = ...  # TODO: pooled covariance
Si = np.linalg.inv(S); logpi = np.log(np.bincount(y[tr]) / 100)
def lda(x): return np.argmax([-0.5 * (x - m) @ Si @ (x - m) + lp for m, lp in zip(mus, logpi)])
def centroid(x): return np.argmin([np.sum((x - m) ** 2) for m in mus])
print("LDA accuracy           ", np.mean([lda(x) == t for x, t in zip(X[te], y[te])]))
print("nearest centroid       ", np.mean([centroid(x) == t for x, t in zip(X[te], y[te])]))

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
from sklearn.datasets import load_iris
X, y = load_iris(return_X_y=True)
idx = np.random.default_rng(0).permutation(150); tr, te = idx[:100], idx[100:]
mus = np.array([X[tr][y[tr] == c].mean(0) for c in range(3)])
S = sum(np.cov(X[tr][y[tr] == c].T) * ((y[tr] == c).sum() - 1) for c in range(3)) / (100 - 3)  # pooled covariance
Si = np.linalg.inv(S); logpi = np.log(np.bincount(y[tr]) / 100)
def lda(x): return np.argmax([-0.5 * (x - m) @ Si @ (x - m) + lp for m, lp in zip(mus, logpi)])
def centroid(x): return np.argmin([np.sum((x - m) ** 2) for m in mus])
print("LDA accuracy           ", np.mean([lda(x) == t for x, t in zip(X[te], y[te])]))
print("nearest centroid       ", np.mean([centroid(x) == t for x, t in zip(X[te], y[te])]))

<details><summary><b>Answer and explanation</b></summary>

LDA gets 98% here; nearest centroid only 86%. The difference is the covariance: petal length and width rise together, so distances must be measured in units that respect that correlation (Mahalanobis, §3.2.2). Nearest centroid is exactly LDA with Σ = I, so it measures plain Euclidean distance and gets fooled (Murphy §9.2.5).

Expected output:
```
LDA accuracy            0.98
nearest centroid        0.86
```

</details>